In [51]:
import import_ipynb
from BollingerBandsIndicator import BollingerBandsIndicator
from EMAIndicator import EMAIndicator
from RSIIndicator import RSIIndicator
import pandas as pd
import datetime
from ResearchClass import StrategyTemplate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from itertools import combinations
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from ResearchClass import PlotEvaluations, EvaluationMetrics, TradesBook


In [52]:
class MLModel(StrategyTemplate):
    ticker_list=['data/MNQc1.parquet']
    
    data = pd.read_parquet(ticker_list)

    # Define parameters for the indicators
    rsi_parameters = [14, 0.02]  # Lookback period of 14, stop-loss percentage 2%
    ema_parameters = [20, 0.02]  # Lookback period of 20, stop-loss percentage 2%
    bollinger_parameters = [20, 2, 0.015]  # Lookback, Std Dev multiplier, Stop-loss

    today = datetime.datetime.now()
    start_date = str((today - datetime.timedelta(days=59)).strftime("%Y-%m-%d"))
    end_date = str(today.strftime("%Y-%m-%d"))
    interval = "1m"
    # Create indicator objects

    data['Returns'] = data['Close'].pct_change()

    # List of indicators to test
    indicators = ['RSI', 'EMA', 'Bollinger Bands']

    # Generate all combinations of indicators
    indicator_combinations = []
    for r in range(1, len(indicators) + 1):
        indicator_combinations.extend(combinations(indicators, r))
    
    def evaluate_combination(data, combination, parameters):
        """
        Evaluates a specific combination of indicators on the dataset.

        Parameters:
        - data: The stock data.
        - combination: A tuple of selected indicators.
        - parameters: A dictionary of parameters for each indicator.

        Returns:
        - Performance metric (e.g., total return).
        """
        # Reset data for fresh calculation
        data = data.copy()

        # Apply selected indicators
        if 'RSI' in combination:
            rsi = RSIIndicator(data, start_date, end_date, interval, parameters['RSI'])
            rsi.AddIndicators()
            data['RSI'] = rsi.data['RSI']

        if 'EMA' in combination:
            ema = EMAIndicator(data, start_date, end_date, interval, parameters['EMA'])
            ema.AddIndicators()
            data['EMA'] = ema.data['EMA']

        if 'Bollinger Bands' in combination:
            bollinger = BollingerBandsIndicator(data, start_date, end_date, interval, parameters['Bollinger Bands'])
            bollinger.AddIndicators()
            data['BB_UPPER'] = bollinger.data['BB_UPPER']
            data['BB_LOWER'] = bollinger.data['BB_LOWER']
            data['BB_MIDDLE'] = bollinger.data['BB_MIDDLE']

        # Implement a simple strategy for testing (e.g., buy/sell logic)
        data['Signal'] = 0
        if 'RSI' in combination:
            data.loc[data['RSI'] < 30, 'Signal'] = 1  # Buy
            data.loc[data['RSI'] > 70, 'Signal'] = -1  # Sell

        if 'EMA' in combination:
            data.loc[data['Close'] > data['EMA'], 'Signal'] = 1  # Buy
            data.loc[data['Close'] < data['EMA'], 'Signal'] = -1  # Sell

        if 'Bollinger Bands' in combination:
            data.loc[data['Close'] < data['BB_LOWER'], 'Signal'] = 1  # Buy
            data.loc[data['Close'] > data['BB_UPPER'], 'Signal'] = -1  # Sell

        # Calculate returns
        data['Daily_Return'] = data['Signal'].shift(1) * data['Close'].pct_change()
        total_return = data['Daily_Return'].sum()

        return total_return

    # Example parameters for indicators
    parameters = {
        'RSI': [14, 0.02],  # Lookback period, stop-loss percentage
        'EMA': [20, 0.02],  # Lookback period, stop-loss percentage
        'Bollinger Bands': [20, 2, 0.015]  # Lookback period, Std Dev multiplier, stop-loss
    }

    # Evaluate all combinations
    best_combination = None
    best_performance = float('-inf')

    for combination in indicator_combinations:
        performance = evaluate_combination(data, combination, parameters)
        print(f"Combination: {combination}, Performance: {performance:.2f}")
        if performance > best_performance:
            best_performance = performance
            best_combination = combination

    print(f"Best Combination: {best_combination}, Best Performance: {best_performance:.2f}")

    # Use the best combination
    for indicator in best_combination:
        if indicator == 'RSI':
            rsi = RSIIndicator(data, start_date, end_date, interval, parameters['RSI'])
            rsi.AddIndicators()
            data['RSI'] = rsi.data['RSI']

        if indicator == 'EMA':
            ema = EMAIndicator(data, start_date, end_date, interval, parameters['EMA'])
            ema.AddIndicators()
            data['EMA'] = ema.data['EMA']

        if indicator == 'Bollinger Bands':
            bollinger = BollingerBandsIndicator(data, start_date, end_date, interval, parameters['Bollinger Bands'])
            bollinger.AddIndicators()
            data['BB_UPPER'] = bollinger.data['BB_UPPER']
            data['BB_LOWER'] = bollinger.data['BB_LOWER']
            data['BB_MIDDLE'] = bollinger.data['BB_MIDDLE']

    # Prepare dataset for the ML model
    data_cleaned=data.fillna(0)
    X = data_cleaned[[col for col in data.columns if col in best_combination]]
    y = data_cleaned['Returns']  # Target variable

        # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=16)

    # Initialize and train the random forest regressor
    rf_regressor = RandomForestRegressor(random_state=16)
    rf_regressor.fit(X_train, y_train)

    # Make predictions
    y_pred = rf_regressor.predict(X_test)

    # Calculate baseline MSE using the mean
    y_mean = np.full_like(y_test, y_test.mean())
    baseline_mse = mean_squared_error(y_test, y_mean)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Mean Squared Error: {mse:.2f}")
    print(f"R² Score: {r2:.2f}")

ticker_list = ['data/MNQc1.parquet']

today = datetime.datetime.now()
start_date = str((today - datetime.timedelta(days=59)).strftime("%Y-%m-%d"))
end_date = str(today.strftime("%Y-%m-%d"))
interval = "1m"

BB_TRADE_BOOK = MLModel(ticker_list, today, start_date, end_date, interval).ApplyStrategyThroughTickers()

#Evaluate the strategy performance
bb_metrics = EvaluationMetrics(BB_TRADE_BOOK, ticker_list, start_date, end_date)
bb_plots = PlotEvaluations(BB_TRADE_BOOK, (10, 7))

#Print the evaluation metrics and plot the summary
bb_metrics.print_results()
bb_plots.PlotSummary()


        
        



Combination: ('RSI',), Performance: 0.30
Combination: ('EMA',), Performance: -1.34
Combination: ('Bollinger Bands',), Performance: 0.29
Combination: ('RSI', 'EMA'), Performance: -1.34
Combination: ('RSI', 'Bollinger Bands'), Performance: 0.44
Combination: ('EMA', 'Bollinger Bands'), Performance: -0.75
Combination: ('RSI', 'EMA', 'Bollinger Bands'), Performance: -0.75
Best Combination: ('RSI', 'Bollinger Bands'), Best Performance: 0.44


KeyboardInterrupt: 